In [51]:
import os
import cv2
import shutil
import numpy as np
import matplotlib.pyplot as plt

In [12]:
def variance_of_laplacian(image, ksize=13):
    '''
    Input: image array (gray)
    Output: variance of Laplacian
    '''
    var = cv2.Laplacian(image, cv2.CV_64F, ksize=ksize).var()
    return var

In [48]:
def fft_blur_detection(image, size, thresh, vis=False):
    h, w = image.shape
    cX, cY = int(w/2.), int(h/2.)
    
    fft = np.fft.fft2(image)
    fftShift = np.fft.fftshift(fft)
    
    if vis:
        # compute the magnitude spectrum of the transform
        magnitude = 20 * np.log(np.abs(fftShift))
        # display the original input image
        (fig, ax) = plt.subplots(1, 2, )
        ax[0].imshow(image, cmap="gray")
        ax[0].set_title("Input")
        ax[0].set_xticks([])
        ax[0].set_yticks([])
        # display the magnitude image
        ax[1].imshow(magnitude, cmap="gray")
        ax[1].set_title("Magnitude Spectrum")
        ax[1].set_xticks([])
        ax[1].set_yticks([])
        # show our plots
        plt.show()
    
    fftShift[cY - size:cY + size, cX - size:cX + size] = 0
    fftShift = np.fft.ifftshift(fftShift)
    recon = np.fft.ifft2(fftShift)
    
    magnitude = 20 * np.log(np.abs(recon))
    mean = np.mean(magnitude)
    
    return mean, (mean<=thresh)

In [108]:
img_root = '/Users/hanruchen/Desktop/BioInfo_LAB/Multi_omics/PARADIGM/kmeans_attention_Pix2Pathway/'
slide = 'TCGA-78-7540-01Z-00-DX1.25c29857-0d5f-4e52-be0c-32c6686b653f'

In [109]:
for img in os.listdir(os.path.join(img_root, slide)):
    try:
        norm_img = cv2.imread(os.path.join(img_root, slide, img))
        norm_img = cv2.cvtColor(norm_img, cv2.COLOR_BGR2GRAY)
        mean, blur = fft_blur_detection(norm_img, 70, 14)
        fm = variance_of_laplacian(norm_img, 13)
        if blur and (fm<4e14 or fm>6e14):
            print(img, mean, fm)
            shutil.copy(os.path.join(img_root, slide, img), os.path.join(img_root, 'blur_img', slide))
            #plt.imshow(cv2.imread(os.path.join(img_root, slide, img)))
            #break
    except:
        print(img)

.DS_Store


In [ ]:
for img in os.listdir(os.path.join(img_root, slide)):
    norm_img = cv2.imread(os.path.join(img_root, slide, img))
    norm_img = cv2.cvtColor(norm_img, cv2.COLOR_BGR2GRAY)
    fm = variance_of_laplacian(norm_img, 9)
    if fm < 4e14 or fm > 6e14:
        shutil.move(os.path.join(img_root, slide, img), os.path.join(img_root, 'blur_img'))
        #plt.imshow(cv2.imread(os.path.join(img_root, slide, img)))
        #break